In [0]:
# =============================================================================
# Notebook  : _bronze_common
# Location  : /HGI-Lakehouse-Pipeline/02_Bronze_Layer/_bronze_common
# Purpose   : All shared logic for Landing Zone → Bronze layer.
#             Every bronze object notebook does:
#               Cell 1: %run ../config/pipeline_config
#               Cell 2: %run ./_bronze_common
#
# REQUIRES pipeline_config to be %run FIRST — uses:
#   SPARK_CONF_BRONZE, DELTA_TBLPROPS_BRONZE, OBJECT_CASING, BRONZE_DDL_COLUMNS,
#   bronze_catalog, landing_path(), checkpoint_path(), bronze_table()
# =============================================================================

import time
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, TimestampType

# =============================================================================
# 0. Serverless-safe Spark config
#    Silently skips: spark.databricks.*, spark.hadoop.fs.s3a.*,
#                   spark.sql.shuffle.partitions
#    (these throw on Serverless but work fine on Job Cluster)
# =============================================================================
def safe_spark_conf(configs: dict):
    """Set spark configs; silently skip any not supported on this compute type."""
    for k, v in configs.items():
        try:
            spark.conf.set(k, v)
        except Exception:
            pass   # Serverless rejects some configs — completely safe to ignore

safe_spark_conf(SPARK_CONF_BRONZE)   # from pipeline_config

# =============================================================================
# 1. Timestamp Repair
#    Guards against corrupted 4.53e25 values in Salesforce SystemModstamp.
#    Tries all 4 valid formats; falls back to NULL (never crashes the batch).
# =============================================================================
# =============================================================================
# 1. Timestamp Repair
#    Guards against corrupted values. 
#    Casts to string first to prevent DATE -> BIGINT strict typing crashes.
# =============================================================================
# =============================================================================
# 1. Timestamp Repair (ANSI-Safe Regex Version)
#    Guards against corrupted values and strict Databricks 15.4 ANSI errors.
#    Uses Regex to identify the format before attempting to parse it.
# =============================================================================
def safe_timestamp(col_expr):
    str_expr = col_expr.cast("string")
    
    # 1. Matches pure date: "YYYY-MM-DD"
    is_date = str_expr.rlike(r"^\d{4}-\d{2}-\d{2}$")
    
    # 2. Matches timestamps with T or space: "YYYY-MM-DD 14:30:00" or "YYYY-MM-DDTHH..."
    is_timestamp = str_expr.rlike(r"^\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}.*$")
    
    # 3. Matches 10-digit Unix seconds
    is_unix_sec = str_expr.rlike(r"^\d{10}$")
    
    # 4. Matches 13-digit Unix milliseconds
    is_unix_ms = str_expr.rlike(r"^\d{13}$")
    
    return F.when(
        is_date, 
        F.to_timestamp(str_expr, "yyyy-MM-dd")
    ).when(
        is_timestamp, 
        str_expr.cast(TimestampType())
    ).when(
        is_unix_sec, 
        F.to_timestamp(str_expr.cast("long").cast(TimestampType()))
    ).when(
        is_unix_ms, 
        F.to_timestamp((str_expr.cast("long") / 1000).cast(TimestampType()))
    ).otherwise(
        F.lit(None).cast(TimestampType())
    )
    
# def safe_timestamp(col_expr):
#     return F.coalesce(
#         F.to_timestamp(col_expr, "yyyy-MM-dd'T'HH:mm:ss.SSSX"),     # 2026-03-09T14:30:00.000Z
#         F.to_timestamp(col_expr, "yyyy-MM-dd'T'HH:mm:ssX"),          # 2026-03-09T14:30:00Z
#         F.to_timestamp(col_expr, "yyyy-MM-dd HH:mm:ss"),              # 2026-03-09 14:30:00
#         F.when(                                                        # Unix seconds (10 digits)
#             F.length(col_expr.cast("string")) == 10,
#             F.to_timestamp(col_expr.cast("long").cast(TimestampType()))
#         ),
#         F.when(                                                        # Unix ms (13 digits)
#             F.length(col_expr.cast("string")) == 13,
#             F.to_timestamp((col_expr.cast("long") / 1000).cast(TimestampType()))
#         ),
#         col_expr.cast(TimestampType())                                 # direct cast → NULL for 4.53e25
#     )

# =============================================================================
# 2. Domain Extraction
# =============================================================================
def extract_domain_from_website(col_expr):
    """Account: strip protocol + www + path, validate TLD format."""
    stripped = F.lower(
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace(col_expr, r'^https?://', ''),
                r'^www\.', ''
            ),
            r'/.*$', ''
        )
    )
    return F.when(
        col_expr.isNotNull() & F.lower(col_expr).rlike(r'^https?://'),
        F.when(
            stripped.rlike(r'^[a-z0-9][a-z0-9\-]*(\.[a-z0-9\-]+)*\.[a-z]{2,}$'),
            stripped
        ).otherwise(F.lit(None))
    ).otherwise(col_expr)

def extract_domain_from_email(col_expr):
    """Contact/User: split_part(lower(email), '@', 2)."""
    return F.when(
        col_expr.isNotNull() & col_expr.contains("@"),
        F.split(F.lower(col_expr), "@").getItem(1)
    ).otherwise(F.lit(None))

# =============================================================================
# 3. Core Normalization
#    Applies: composite ID, timestamp repair, object-specific field mapping,
#             dynamic a_ prefix for all remaining fields.
# =============================================================================
CORE_TRACKING = frozenset({
    "tenant", "id", "source_system", "source_system_object",
    "source_key_name", "source_key_value", "data_timestamp", "status",
    "domain", "name", "email",
    "contact_source_system_object", "contact_source_key_value",
    "campaign_source_key_value", "event_timestamp",
    "a_accountid", "a_subject", "a_amount", "a_stagename",
    "event", "event_text", "systemmodstamp", "created_date",
    "created_at", "updated_at", "extraction_timestamp",
})

def normalize_batch(df, source_sys: str, obj_name: str,
                    sf_object_name: str, tenant_id: int):
    """Full normalization per PDF spec. Returns DataFrame ready for dedup + merge."""
    raw_cols = [c for c in df.columns if c != "extraction_timestamp"]
    df = df.withColumn("status", F.lit("processing"))

    if source_sys == "salesforce":
        # ── Base tracking columns ─────────────────────────────────────────────
        df = (df
            .withColumn("tenant",               F.lit(tenant_id).cast(LongType()))
            .withColumn("source_system",        F.lit("salesforce"))
            .withColumn("source_system_object", F.lit(sf_object_name))
            .withColumn("source_key_name",      F.lit("Id"))
            .withColumn("source_key_value",     F.col("Id"))
            .withColumn("id",
                F.concat(F.lit(f"salesforce_{sf_object_name}_Id_"), F.col("Id")))
            .withColumn("data_timestamp",  safe_timestamp(F.col("SystemModstamp")))
            .withColumn("systemmodstamp",  safe_timestamp(F.col("SystemModstamp")))
            .withColumn("created_date",    safe_timestamp(F.col("CreatedDate")))
            .withColumn("created_at",      F.current_timestamp())
            .withColumn("updated_at",      F.current_timestamp())
        )
        known = {"id", "systemmodstamp", "createddate", "extraction_timestamp", "isdeleted"}

        # ── Object-specific field mapping ─────────────────────────────────────
        if obj_name == "account":
            known.update({"name", "website", "numberofemployees"})
            df = (df
                .withColumn("name",   F.col("Name"))
                .withColumn("domain", extract_domain_from_website(F.col("Website")))
                .withColumn("a_isdeleted",         F.col("IsDeleted").cast("string"))
                .withColumn("a_website",           F.col("Website").cast("string"))
                .withColumn("a_numberofemployees", F.col("NumberOfEmployees").cast("string"))
            )

        elif obj_name == "contact":
            known.update({"email", "firstname", "lastname", "accountid"})
            df = (df
                .withColumn("email",       F.lower(F.col("Email")))
                .withColumn("domain",      extract_domain_from_email(F.col("Email")))
                .withColumn("a_firstname", F.col("FirstName").cast("string"))
                .withColumn("a_lastname",  F.col("LastName").cast("string"))
                .withColumn("a_accountid", F.col("AccountId").cast("string"))
                .withColumn("a_isdeleted", F.col("IsDeleted").cast("string"))
            )

        elif obj_name == "opportunity":
            known.update({"accountid", "stagename", "amount", "iswon", "isclosed"})
            df = (df
                .withColumn("a_accountid", F.col("AccountId").cast("string"))
                .withColumn("a_stagename", F.col("StageName").cast("string"))
                .withColumn("a_amount",    F.col("Amount").cast("string"))   # numeric as string per spec
                .withColumn("a_iswon",     F.col("IsWon").cast("string"))
                .withColumn("a_isclosed",  F.col("IsClosed").cast("string"))
                .withColumn("a_isdeleted", F.col("IsDeleted").cast("string"))
            )

        elif obj_name == "task":
            known.update({"whoid", "whatid", "activitydate", "subject"})
            df = (df
                .withColumn("contact_source_system_object",
                    F.when(F.col("WhoId").startswith("003"), F.lit("Contact"))
                     .when(F.col("WhoId").startswith("00Q"), F.lit("Lead"))
                     .when(F.col("WhatId").startswith("001"), F.lit("Account"))
                     .when(F.col("WhatId").startswith("006"), F.lit("Opportunity"))
                     .when(F.col("WhatId").startswith("500"), F.lit("Case"))
                     .when(F.col("WhatId").startswith("701"), F.lit("Campaign"))
                     .otherwise(F.lit("Unknown"))
                )
                .withColumn("contact_source_key_value",
                    F.coalesce(F.col("WhoId"), F.col("WhatId")).cast("string"))
                .withColumn("event_timestamp", safe_timestamp(F.col("ActivityDate")))
                .withColumn("a_subject",   F.col("Subject").cast("string"))
                .withColumn("a_isdeleted", F.col("IsDeleted").cast("string"))
            )

        elif obj_name == "campaign":
            known.update({"name", "type"})
            df = (df
                .withColumn("a_name", F.col("Name").cast("string"))
                .withColumn("a_type", F.col("Type").cast("string"))
            )

        elif obj_name == "campaignmember":
            known.update({"campaignid", "leadid", "contactid"})
            df = (df
                .withColumn("campaign_source_key_value", F.col("CampaignId").cast("string"))
                .withColumn("contact_source_key_value",
                    F.coalesce(F.col("ContactId"), F.col("LeadId")).cast("string"))
            )

        elif obj_name == "user":
            known.update({"email", "username", "firstname", "lastname"})
            df = (df
                .withColumn("email",      F.lower(F.col("Email")))
                .withColumn("domain",     extract_domain_from_email(F.col("Email")))
                .withColumn("a_firstname", F.col("FirstName").cast("string"))
                .withColumn("a_lastname",  F.col("LastName").cast("string"))
                .withColumn("a_username",  F.col("Username").cast("string"))
                .withColumn("a_isactive",  F.col("IsActive").cast("string"))
            )

        # ── Dynamic a_ prefixing for all other fields ─────────────────────────
        current_cols = {c.lower() for c in df.columns}
        for c in raw_cols:
            cl = c.lower()
            if cl in known:
                continue
            target_col = cl if cl.startswith("a_") else f"a_{cl}"
            if target_col not in current_cols:
                df = df.withColumn(target_col, F.col(c).cast("string"))
                current_cols.add(target_col)

        # ── Final ordered column selection ────────────────────────────────────
        seen, ordered = set(), []
        def _add(n):
            if n.lower() not in seen and n in df.columns:
                ordered.append(n)
                seen.add(n.lower())

        for c in ["tenant", "id", "source_system", "source_system_object",
                  "source_key_name", "source_key_value", "data_timestamp",
                  "created_date", "created_at", "updated_at", "status",
                  "systemmodstamp", "extraction_timestamp"]:
            _add(c)
        for c in ["domain", "name", "email", "contact_source_system_object",
                  "contact_source_key_value", "campaign_source_key_value",
                  "event_timestamp"]:
            _add(c)
        for c in sorted(df.columns):
            if c.startswith("a_"):
                _add(c)
        df = df.select(*ordered)

    elif source_sys == "bigquery":
        # BQ events — stored in events_raw table
        df = (df
            .withColumn("tenant",               F.lit(tenant_id).cast(LongType()))
            .withColumn("source_system",        F.lit("bigquery"))
            .withColumn("source_system_object", F.lit("Event"))
            .withColumn("source_key_name",      F.lit("event_id"))
            .withColumn("source_key_value",     F.col("id"))
            .withColumn("id",
                F.concat(F.lit("bigquery_Event_event_id_"), F.col("id")))
            .withColumn("data_timestamp",  safe_timestamp(F.col("event_timestamp")))
            .withColumn("created_date",    safe_timestamp(F.col("event_timestamp")))
            .withColumn("created_at",      F.current_timestamp())
            .withColumn("updated_at",      F.current_timestamp())
        )

    return df

def get_hash_cols(df) -> list:
    """Returns columns to include in xxhash64 fingerprint for change detection."""
    core_attr = {
        "domain", "name", "email",
        "contact_source_system_object", "contact_source_key_value",
        "campaign_source_key_value", "event_timestamp", "data_timestamp",
    }
    return [c for c in df.columns if c.startswith("a_") or c in core_attr]

# =============================================================================
# 4. Schema Evolution  (driver-level only — never inside foreachBatch executor)
# =============================================================================
def evolve_schema_if_needed(new_schema, table_full: str):
    """Compare incoming schema with target; ADD COLUMNS for any new fields."""
    existing = {f.name.lower() for f in spark.table(table_full).schema}
    new_cols = [
        f"`{f.name}` {f.dataType.simpleString()}"
        for f in new_schema
        if f.name.lower() not in existing and not f.name.startswith("_")
    ]
    if new_cols:
        print(f"  Schema evolution ({table_full}): +{len(new_cols)} col(s): {new_cols[:3]}...")
        spark.sql(f"ALTER TABLE {table_full} ADD COLUMNS ({', '.join(new_cols)})")

# =============================================================================
# 5. foreachBatch Handler
#    Steps: Normalize → Intra-batch dedup → xxhash64 → Schema evolve →
#           Cross-batch change detect → COALESCE MERGE
# =============================================================================
def process_bronze_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return
    active_spark = batch_df.sparkSession

    # A. Normalize (composite ID, timestamps, domain extraction, a_ columns)
    df_norm = normalize_batch(batch_df, source_system, object_name,
                              sf_object_name, tenant_id)

    # B. Intra-batch dedup: sort DESC by extraction_timestamp then dropDuplicates
    #    Avoids the Window function shuffle — same result, much faster
    dedup_keys = ["source_system", "source_system_object",
                  "source_key_name", "source_key_value"]
    if "extraction_timestamp" in df_norm.columns:
        df_deduped = (
            df_norm
            .sort(F.col("extraction_timestamp").desc(), F.col("data_timestamp").desc())
            .dropDuplicates(dedup_keys)
            .drop("extraction_timestamp")
        )
    else:
        df_deduped = df_norm.dropDuplicates(dedup_keys)

    # C. Record hash for cross-batch change detection
    hash_cols = get_hash_cols(df_deduped)
    df_deduped = df_deduped.withColumn(
        "record_hash",
        F.xxhash64(*[F.col(c) for c in hash_cols]).cast("string")
        if hash_cols else F.lit(None).cast("string")
    )

    # D. Schema evolution (driver-level — safe)
    try:
        evolve_schema_if_needed(df_deduped.schema, table_full_name)
    except Exception as e:
        print(f"  Schema evolution warning (non-fatal): {e}")

    # E. Cross-batch change detection — only changed/new rows go to MERGE
    try:
        target_hashes = (
            active_spark.table(table_full_name)
                        .select("id", F.col("record_hash").alias("t_hash"))
        )
        df_changed = (
            df_deduped
            .join(target_hashes, on="id", how="left")
            .filter(F.col("t_hash").isNull() | (F.col("record_hash") != F.col("t_hash")))
            .drop("t_hash")
        )
    except Exception:
        df_changed = df_deduped   # table is empty / first run

    if df_changed.isEmpty():
        print(f"  Batch {batch_id}: no changes detected, skipping MERGE.")
        return

    # F. COALESCE MERGE (= LAST_VALUE IGNORE NULLS per spec)
    
    # NEW FIX: Drop internal Databricks columns before the merge
    drop_cols = [c for c in df_changed.columns if c.startswith("_")]
    df_changed = df_changed.drop(*drop_cols)

    view_name = f"bronze_{object_name}_{batch_id}_{int(time.time())}"
    df_changed.createOrReplaceTempView(view_name)

    cols = df_changed.columns
    
    pk_cols        = {"tenant", "id", "source_system", "source_system_object",
                      "source_key_name", "source_key_value", "created_at", "status"}
    always_overwrite = {"updated_at", "data_timestamp", "record_hash"}

    update_parts = []
    for c in cols:
        if c in pk_cols:
            continue
        if c in always_overwrite:
            update_parts.append(f"target.`{c}` = source.`{c}`")
        else:
            # COALESCE keeps last known non-null value if source sends NULL
            update_parts.append(
                f"target.`{c}` = COALESCE(source.`{c}`, target.`{c}`)"
            )
    update_set = ", ".join(update_parts) + ", target.`status` = 'updated'"

    ins_cols = ", ".join(f"`{c}`" for c in cols)
    ins_vals = ", ".join(
        f"source.`{c}`" if c != "status" else "'new'" for c in cols
    )

    # Timestamp guard: prevents stale out-of-order batches overwriting fresher data
    active_spark.sql(f"""
        MERGE INTO {table_full_name} AS target
        USING {view_name} AS source
        ON  target.source_system        = source.source_system
        AND target.source_system_object = source.source_system_object
        AND target.source_key_name      = source.source_key_name
        AND target.source_key_value     = source.source_key_value
        WHEN MATCHED
          AND source.data_timestamp > COALESCE(target.data_timestamp,
              CAST('1900-01-01' AS TIMESTAMP))
        THEN UPDATE SET {update_set}
        WHEN NOT MATCHED THEN INSERT ({ins_cols}) VALUES ({ins_vals})
    """)
    print(f"  Batch {batch_id}: MERGE complete → {table_full_name}")

# =============================================================================
# 6. Table Creation Helper
# =============================================================================
def create_bronze_table(table_full: str, obj_name: str):
    """Creates the bronze Delta table if it does not exist."""
    schema_def = BRONZE_DDL_COLUMNS[obj_name]   # from pipeline_config
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table_full} (
            {schema_def}
        )
        USING DELTA
        CLUSTER BY (source_system, source_system_object, source_key_name, source_key_value)
        {DELTA_TBLPROPS_BRONZE}
    """)
    print(f"  Bronze table ready: {table_full}")

# =============================================================================
# 7. Auto Loader Stream Launcher
# =============================================================================
def start_ingestion():
    """Reads parquet from landing zone, writes to bronze via foreachBatch."""
    print(f"Auto Loader: {landing_zone_path} → {table_full_name}")
    df_stream = (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format",             "parquet")
             .option("cloudFiles.inferColumnTypes",    "true")
             .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
             .option("cloudFiles.schemaLocation",      schema_loc)
             .option("cloudFiles.maxFilesPerTrigger",  str(max_files_per_trigger))
             .option("cloudFiles.useNotifications",    "false")
             .load(landing_zone_path)
             .withColumn("extraction_timestamp",
                         F.col("_metadata.file_modification_time"))
    )
    query = (
        df_stream.writeStream
                 .foreachBatch(process_bronze_batch)
                 .option("checkpointLocation", checkpoint_loc)
                 .trigger(availableNow=True)    # batch mode: process all then stop
                 .start()
    )
    query.awaitTermination()
    print(f"  Stream complete: {object_name}")

# =============================================================================
# 8. Retry Wrapper
# =============================================================================
def run_with_retry(fn, retries=3, delay_secs=15):
    for attempt in range(1, retries + 1):
        try:
            return fn()
        except Exception as e:
            print(f"  Attempt {attempt}/{retries} failed: {e}")
            if attempt == retries:
                raise
            time.sleep(delay_secs)

print("_bronze_common loaded.")
